In [ ]:
import csv
import re
from collections import Counter
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
import zipfile
import os

zip_path = "/content/IMDB Dataset.csv.zip"
extract_path = "/content/IMDB"

try:
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    print("Extracted files:", os.listdir(extract_path))
except FileNotFoundError:
    print(f"Error: The file '{zip_path}' was not found.")
    print("Please upload the zip file to your Colab environment and update the 'zip_path' variable.")
except Exception as e:
    print(f"An error occurred: {e}")

Extracted files: ['IMDB Dataset.csv']


In [ ]:
import csv

csv_path = "/content/IMDB/IMDB Dataset.csv"

reviews, labels = [], []

with open(csv_path, encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        reviews.append(row["review"])
        labels.append(1 if row["sentiment"] == "positive" else 0)

print("Total samples:", len(reviews))
print("First review:", reviews[0][:200])
print("First label:", labels[0])

Total samples: 50000
First review: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo
First label: 1


In [ ]:
import re
from collections import Counter
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Dataset, random_split
import torch.optim as optim


def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z]", " ", text)
    words = text.split()
    return words

all_words = []
for r in reviews:
    all_words.extend(clean_text(r))

from collections import Counter
word_count = Counter(all_words)

max_vocab = 20000
vocab = {"<PAD>": 0, "<UNK>": 1}
for i, (w, _) in enumerate(word_count.most_common(max_vocab)):
    vocab[w] = i + 2

def encode(text, maxlen=200):
    words = clean_text(text)
    nums = [vocab.get(w, 1) for w in words]
    if len(nums) < maxlen:
        nums = nums + [0]*(maxlen - len(nums))
    else:
        nums = nums[:maxlen]
    return nums

encoded_reviews = [encode(r) for r in reviews]

class IMDBDataset(Dataset):
  def __init__(self, reviews, labels):
    self.reviews = torch.tensor(reviews, dtype=torch.long)
    self.labels = torch.tensor(labels, dtype=torch.long)
  def __len__(self):
    return len(self.reviews)
  def __getitem__(self, idx):
    return self.reviews[idx], self.labels[idx]

dataset = IMDBDataset(encoded_reviews, labels)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)


class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super(SentimentRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.embedding(x)
        _, (h, _) = self.rnn(x)
        h = self.dropout(h[-1])
        return self.fc(h)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocab_size = len(vocab)
embed_dim = 128
hidden_dim = 128
output_dim = 2

model = SentimentRNN(vocab_size, embed_dim, hidden_dim, output_dim).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5
for epoch in range(epochs):
    model.train()
    total_correct = 0
    total_samples = 0
    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()


        preds = torch.argmax(outputs, dim=1)
        total_correct += (preds == y).sum().item()
        total_samples += y.size(0)

        if (batch_idx+1) % 100 == 0:
            acc = 100.0 * total_correct / total_samples
            print(f"Epoch [{epoch+1}/{epochs}], Step [{batch_idx+1}/{len(train_loader)}], "
                  f"Loss: {loss.item():.4f}, Training Accuracy: {acc:.2f}%")

model.eval()
correct, total = 0, 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        outputs = model(x)
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

print(f"\nFinal Test Accuracy: {100.0 * correct / total:.2f}%")

Epoch [1/5], Step [100/625], Loss: 0.6799, Training Accuracy: 50.45%
Epoch [1/5], Step [200/625], Loss: 0.6900, Training Accuracy: 50.30%
Epoch [1/5], Step [300/625], Loss: 0.6860, Training Accuracy: 50.88%
Epoch [1/5], Step [400/625], Loss: 0.6868, Training Accuracy: 51.11%
Epoch [1/5], Step [500/625], Loss: 0.6753, Training Accuracy: 51.45%
Epoch [1/5], Step [600/625], Loss: 0.7094, Training Accuracy: 51.66%
Epoch [2/5], Step [100/625], Loss: 0.6754, Training Accuracy: 55.86%
Epoch [2/5], Step [200/625], Loss: 0.6724, Training Accuracy: 56.01%
Epoch [2/5], Step [300/625], Loss: 0.6678, Training Accuracy: 55.77%
Epoch [2/5], Step [400/625], Loss: 0.6815, Training Accuracy: 54.92%
Epoch [2/5], Step [500/625], Loss: 0.7019, Training Accuracy: 54.68%
Epoch [2/5], Step [600/625], Loss: 0.6735, Training Accuracy: 54.82%
Epoch [3/5], Step [100/625], Loss: 0.6447, Training Accuracy: 56.95%
Epoch [3/5], Step [200/625], Loss: 0.6226, Training Accuracy: 59.64%
Epoch [3/5], Step [300/625], Loss:

In [ ]:
import random

sample_indices = random.sample(range(len(test_dataset)), 5)

for i, idx in enumerate(sample_indices):
    x, y = test_dataset[idx]
    x = x.unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(x)
        pred = torch.argmax(outputs, dim=1).item()


    inv_vocab = {v:k for k,v in vocab.items()}
    review_words = [inv_vocab.get(token.item(), "<UNK>") for token in x[0] if token.item() != 0]
    review_text = " ".join(review_words)


    print(f"Sample {i+1}:")
    print(f"Review: {review_text}")
    print(f"True Label: {'Positive' if y.item()==1 else 'Negative'}")
    print(f"Predicted Label: {'Positive' if pred==1 else 'Negative'}\n")


Sample 1:
Review: i have been watching movies from i think last years and i must say that i never felt that bad which i felt after watching this extra large bore movie it was bad very bad there were songs songs nobody should watch this movie the director has shown germans speaking english which is so rubbish germans does not speak english in one scene there was a white girl who asked himesh for autograph <UNK> that he must have gave some money to her in the promo they have shown prepare for laughing riot but i could say there was only one scene where that himesh was laughing for no reason may be he thinks he s funny <UNK> is very good she is like an angel but too young only yrs old if you have plenty of time and don t know what to do then you should watch this movie or else its waste of money
True Label: Negative
Predicted Label: Negative

Sample 2:
Review: this movie was so stupid i couldn t believe what i was seeing as i was watching it it was like a huge train wreck i couldn t look 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

vocab_size = len(vocab)
embed_dim = 64
hidden_dim = 128
output_dim = 2
seq_len = 200
lr = 0.001
epochs = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

reviews_tensor = torch.tensor(encoded_reviews, dtype=torch.long)
labels_tensor = torch.tensor(labels, dtype=torch.long)
dataset = TensorDataset(reviews_tensor, labels_tensor)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)


class SimpleRNN:
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, seq_len):

        self.embeddings = torch.randn(vocab_size, embed_dim, requires_grad=True, device=device)

        self.Wx = torch.randn(embed_dim, hidden_dim, requires_grad=True, device=device)
        self.Wh = torch.randn(hidden_dim, hidden_dim, requires_grad=True, device=device)
        self.bh = torch.zeros(hidden_dim, requires_grad=True, device=device)
        self.Wo = torch.randn(hidden_dim, output_dim, requires_grad=True, device=device)
        self.bo = torch.zeros(output_dim, requires_grad=True, device=device)
        self.hidden_dim = hidden_dim

    def forward(self, x):
        batch_size = x.size(0)
        h = torch.zeros(batch_size, self.hidden_dim, device=device)
        for t in range(x.size(1)):
            xt = self.embeddings[x[:, t]]
            h = torch.tanh(xt @ self.Wx + h @ self.Wh + self.bh)
        logits = h @ self.Wo + self.bo
        return logits

    def parameters(self):
        return [self.embeddings, self.Wx, self.Wh, self.bh, self.Wo, self.bo]

model = SimpleRNN(vocab_size, embed_dim, hidden_dim, output_dim, seq_len)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

for epoch in range(epochs):
    total_correct, total_samples = 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = model.forward(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        preds = torch.argmax(outputs, dim=1)
        total_correct += (preds == y).sum().item()
        total_samples += y.size(0)
    acc = 100 * total_correct / total_samples
    print(f"Epoch {epoch+1}/{epochs}, Training Accuracy: {acc:.2f}%")
total_correct, total_samples = 0, 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        outputs = model.forward(x)
        preds = torch.argmax(outputs, dim=1)
        total_correct += (preds == y).sum().item()
        total_samples += y.size(0)
print(f"Test Accuracy: {100*total_correct/total_samples:.2f}%")


Epoch 1/10, Training Accuracy: 49.98%
Epoch 2/10, Training Accuracy: 49.98%
Epoch 3/10, Training Accuracy: 49.98%
Epoch 4/10, Training Accuracy: 49.98%
Epoch 5/10, Training Accuracy: 49.98%
Epoch 6/10, Training Accuracy: 49.98%
Epoch 7/10, Training Accuracy: 49.98%
Epoch 8/10, Training Accuracy: 49.98%
Epoch 9/10, Training Accuracy: 49.98%
Epoch 10/10, Training Accuracy: 49.98%
Test Accuracy: 50.07%
